# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import datetime as dt

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
aviation_accidents_df = pd.read_csv("data/AviationData.csv", encoding="latin-1")

# Check the structure of the dataset
aviation_accidents_df.head() # Display the first few rows of the dataset


C:\Users\Admin\AppData\Local\Temp\ipykernel_22752\1578668952.py:1: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  aviation_accidents_df = pd.read_csv("data/AviationData.csv", encoding="latin-1")


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


In [3]:
# Check the structure of the dataset
aviation_accidents_df.info() # Get a summary of the dataset, including data types and non-null counts


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

Observation: The dataset has 5 numerical and 26 categorical data types

In [4]:
# Get summary statistics for numerical columns
aviation_accidents_df.describe() 

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


In [5]:
# Check for missing values in each column, sorted in descending order
aviation_accidents_df.isna().sum().sort_values(ascending=False) 

Schedule                  76307
Air.carrier               72241
FAR.Description           56866
Aircraft.Category         56602
Longitude                 54516
Latitude                  54507
Airport.Code              38757
Airport.Name              36185
Broad.phase.of.flight     27165
Publication.Date          13771
Total.Serious.Injuries    12510
Total.Minor.Injuries      11933
Total.Fatal.Injuries      11401
Engine.Type                7096
Report.Status              6384
Purpose.of.flight          6192
Number.of.Engines          6084
Total.Uninjured            5912
Weather.Condition          4492
Aircraft.damage            3194
Registration.Number        1382
Injury.Severity            1000
Country                     226
Amateur.Built               102
Model                        92
Make                         63
Location                     52
Accident.Number               0
Investigation.Type            0
Event.Id                      0
Event.Date                    0
dtype: i

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [6]:
# Cleans and standardizes column names by removing extra spaces, converting to lowercase, and replacing spaces and periods with underscores for consistency and easier data handling
aviation_accidents_df.columns = (
    aviation_accidents_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('.', '_', regex=False)
)

aviation_accidents_df.columns.tolist()

['event_id',
 'investigation_type',
 'accident_number',
 'event_date',
 'location',
 'country',
 'latitude',
 'longitude',
 'airport_code',
 'airport_name',
 'injury_severity',
 'aircraft_damage',
 'aircraft_category',
 'registration_number',
 'make',
 'model',
 'amateur_built',
 'number_of_engines',
 'engine_type',
 'far_description',
 'schedule',
 'purpose_of_flight',
 'air_carrier',
 'total_fatal_injuries',
 'total_serious_injuries',
 'total_minor_injuries',
 'total_uninjured',
 'weather_condition',
 'broad_phase_of_flight',
 'report_status',
 'publication_date']

In [7]:
# Inspect relevant columns with missing values

aviation_accidents_df[['aircraft_category', 'amateur_built', 'event_date']].head()

aviation_accidents_df[['aircraft_category', 'amateur_built']].value_counts(dropna=False)

aviation_accidents_df['event_date'].head()

0    1948-10-24
1    1962-07-19
2    1974-08-30
3    1977-06-19
4    1979-08-02
Name: event_date, dtype: object

Observations from Data Inspection
The dataset contains key fields such as aircraft category, amateur-built status, and event date, which are useful for analyzing aviation accident trends.
There are missing values present in aircraft_category and amateur_built, indicating incomplete records that may need cleaning before analysis.
The combination of value_counts(dropna=False) shows that missing values are explicitly included in the dataset and should not be ignored.
The event_date column is in string format and spans multiple decades (e.g., 1948–1979 in the sample), suggesting long-term historical coverage.
Overall, the dataset requires data type conversion and missing value handling to ensure accurate time-based and category-based analysis.

In [8]:
# Check missing values in the specific columns of interest
aviation_accidents_df[['aircraft_category', 'amateur_built', 'event_date']].isna().sum() 

aircraft_category    56602
amateur_built          102
event_date               0
dtype: int64

The dataset contains missing values in key fields (aircraft_category, amateur_built, and event_date), confirming that data cleaning is required before analysis. These gaps could affect trend and category-based insights if not properly handled.

In [9]:
# Handle missing values in 'Aircraft.Category' and 'Amateur.Built' by filling them with 'Unknown'
aviation_accidents_df['aircraft_category'] = aviation_accidents_df['aircraft_category'].fillna('Unknown')
aviation_accidents_df['amateur_built'] = aviation_accidents_df['amateur_built'].fillna('Unknown')

# Convert 'Event.Date' to datetime format, handling errors by coercing invalid dates to NaT
aviation_accidents_df['event_date'] = pd.to_datetime(aviation_accidents_df['event_date'], errors='coerce')

# Check the structure of the dataset again after handling missing values
aviation_accidents_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   event_id                88889 non-null  object        
 1   investigation_type      88889 non-null  object        
 2   accident_number         88889 non-null  object        
 3   event_date              88889 non-null  datetime64[ns]
 4   location                88837 non-null  object        
 5   country                 88663 non-null  object        
 6   latitude                34382 non-null  object        
 7   longitude               34373 non-null  object        
 8   airport_code            50132 non-null  object        
 9   airport_name            52704 non-null  object        
 10  injury_severity         87889 non-null  object        
 11  aircraft_damage         85695 non-null  object        
 12  aircraft_category       88889 non-null  object

### Apply Filters

#### Keep only airplanes

In [10]:
# Filter the dataset to include only rows where 'aircraft_category' is 'Airplane'
aviation_accidents_df = aviation_accidents_df[
    aviation_accidents_df['aircraft_category'] == 'Airplane'
]

#### Keep only professionally built aircraft

In [11]:
# Filter the dataset to include only rows where 'amateur_built' is 'No'
aviation_accidents_df = aviation_accidents_df[
    aviation_accidents_df['amateur_built'] == 'No'
]

#### Keep only last 40 years (1983 onwards)

In [69]:
# Filter the dataset to include only rows where 'event_date' is in the year 1983 or later
aviation_accidents_df = aviation_accidents_df[
    aviation_accidents_df['event_date'].dt.year >= 1983
]

Conclusive Observation

The dataset has been filtered to improve analysis quality by focusing only on non-amateur-built airplane accidents from 1983 onward. This ensures the dataset is more consistent, relevant, and focused on modern commercial/general aviation trends, while removing unrelated aircraft types, older records, and amateur-built cases that could distort the analysis.

#### Verify Filtering

In [70]:
# Check the structure of the dataset again after filtering
aviation_accidents_df.info() 
aviation_accidents_df.shape # Get the number of rows and columns in the dataset
aviation_accidents_df['event_date'].dt.year.min() # Get the earliest year in the 'event_date' column (1983)
aviation_accidents_df['event_date'].dt.year.max() # Get the latest year in the 'event_date' column (2022)

<class 'pandas.core.frame.DataFrame'>
Index: 17067 entries, 4150 to 88886
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   event_id                17067 non-null  object        
 1   investigation_type      17067 non-null  object        
 2   accident_number         17067 non-null  object        
 3   event_date              17067 non-null  datetime64[ns]
 4   location                17063 non-null  object        
 5   country                 17066 non-null  object        
 6   latitude                15782 non-null  object        
 7   longitude               15779 non-null  object        
 8   airport_code            11432 non-null  object        
 9   airport_name            11531 non-null  object        
 10  injury_severity         17067 non-null  object        
 11  aircraft_damage         16403 non-null  object        
 12  aircraft_category       17067 non-null  object  

np.int32(2022)

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [14]:
# Display the first few rows of the injury columns to check for missing values and get an overview of the data
aviation_accidents_df[[
    'total_fatal_injuries',
    'total_serious_injuries',
    'total_minor_injuries',
    'total_uninjured'
]].head() # Display the first few rows of the injury columns


,total_fatal_injuries,total_serious_injuries,total_minor_injuries,total_uninjured
4149,NaN,NaN,NaN,588.0
4150,NaN,NaN,NaN,588.0
4171,1.0,1.0,NaN,NaN
4285,1.0,NaN,NaN,4.0
5957,NaN,NaN,NaN,289.0


In [15]:
# Check for missing values in the injury columns
# This is important to ensure that we have complete data for our analysis of injuries in aviation accidents.
aviation_accidents_df[[
    'total_fatal_injuries',
    'total_serious_injuries',
    'total_minor_injuries',
    'total_uninjured'
]].isna().sum() # Check for missing values in the injury columns

total_fatal_injuries      2750
total_serious_injuries    2828
total_minor_injuries      2544
total_uninjured            711
dtype: int64

### Handle missing values

The dataset contains multiple columns describing injury outcomes, including fatal, serious, minor, and uninjured passengers. However, missing values are present across these columns.

Since these variables represent counts, missing values are assumed to indicate zero injuries/uninjured passengers rather than unknown values. Therefore, all missing values are imputed with 0.

In [16]:
injury_cols = [
    'total_fatal_injuries',
    'total_serious_injuries',
    'total_minor_injuries',
    'total_uninjured'
]

aviation_accidents_df[injury_cols] = aviation_accidents_df[injury_cols].fillna(0)

In [17]:
# Check for missing values in the injury columns again after filling missing values with 0
aviation_accidents_df[[
    'total_fatal_injuries',
    'total_serious_injuries',
    'total_minor_injuries',
    'total_uninjured'
]].head()

,total_fatal_injuries,total_serious_injuries,total_minor_injuries,total_uninjured
4149,0.0,0.0,0.0,588.0
4150,0.0,0.0,0.0,588.0
4171,1.0,1.0,0.0,0.0
4285,1.0,0.0,0.0,4.0
5957,0.0,0.0,0.0,289.0


A new variable, total_passengers, was created to estimate the total number of individuals involved in each accident. This is calculated as the sum of all injury categories.

This metric is essential because it allows injury severity to be analyzed proportionally rather than using raw counts, which would bias results toward larger aircraft.

In [18]:
# Constrct a new column 'total_passengers' by summing the injury columns
aviation_accidents_df['total_passengers'] = (
    aviation_accidents_df['total_fatal_injuries'] +
    aviation_accidents_df['total_serious_injuries'] +
    aviation_accidents_df['total_minor_injuries'] +
    aviation_accidents_df['total_uninjured']
)

In [19]:
# Construct Serious + Fatal Injuries
aviation_accidents_df['serious_fatal_injuries'] = (
    aviation_accidents_df['total_fatal_injuries'] +
    aviation_accidents_df['total_serious_injuries']
)

### Remove invalid rows

In [20]:
# Filter the dataset to include only rows where 'total_passengers' is greater than 0
aviation_accidents_df = aviation_accidents_df[
    aviation_accidents_df['total_passengers'] > 0
]

### Create Injury Rate (KEY METRIC)

In [21]:
aviation_accidents_df['injury_rate'] = (
    aviation_accidents_df['serious_fatal_injuries'] /
    aviation_accidents_df['total_passengers']
)

The primary safety metric constructed is injury_rate, defined as the proportion of passengers who experienced serious or fatal injuries in an accident.

This is calculated as:

\text{injury_rate} = \frac{\text{fatal injuries + serious injuries}}{\text{total passengers}}

This metric provides a normalized measure of accident severity, allowing fair comparisons across aircraft of different sizes. It directly aligns with the client’s interest in minimizing severe injury outcomes.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

The aircraft_damage column describes the extent of damage sustained during an accident. For the purpose of this analysis, the client is particularly interested in whether an aircraft is destroyed or not, as this reflects structural robustness.

In [22]:
# Inspect the 'aircaft_damage' column to understand the categories and their distribution
aviation_accidents_df['aircraft_damage'].value_counts(dropna=False)

aircraft_damage
Substantial    16821
Destroyed       2268
NaN              787
Minor            593
Unknown           74
Name: count, dtype: int64

### Construct a derived column tracking whether an aircraft was destroyed or not.

A binary variable, aircraft_destroyed, was created to indicate whether an aircraft was completely destroyed in an accident:

1 → Destroyed
0 → Not destroyed

This simplifies analysis and enables calculation of destruction rates across aircraft types and conditions.

In [23]:
# Create a new binary column 'aircraft_destroyed' where 1 indicates 'Destroyed' and 0 indicates otherwise
aviation_accidents_df['aircraft_destroyed'] = aviation_accidents_df[
    'aircraft_damage'
].apply(lambda x: 1 if x == 'Destroyed' else 0)

In [24]:
# Check the structure of the dataset again after creating new columns
#aviation_accidents_df.info()
aviation_accidents_df[[
    'total_passengers',
    'serious_fatal_injuries',
    'injury_rate',
    'aircraft_destroyed'
]].describe()

,total_passengers,serious_fatal_injuries,injury_rate,aircraft_destroyed
count,20543.000000,20543.000000,20543.000000,20543.000000
mean,8.970404,0.967337,0.283971,0.110403
std,36.645481,6.962195,0.431704,0.313399
min,1.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000,0.000000
75%,2.000000,1.000000,0.800000,0.000000
max,588.000000,295.000000,1.000000,1.000000


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [25]:
# Inspect the make column to understand the distribution of aircraft makes in the dataset
aviation_accidents_df['make'].head()

4149        Lockheed
4150          Boeing
4171           Piper
4285    De Havilland
5957         Douglas
Name: make, dtype: object

In [26]:
# Display the top 20 most common aircraft makes in the dataset
aviation_accidents_df['make'].value_counts().head(20) 

make
CESSNA                4794
PIPER                 2777
Cessna                2270
Piper                 1183
BEECH                 1003
BOEING                 642
Beech                  409
MOONEY                 235
CIRRUS DESIGN CORP     218
AIR TRACTOR INC        217
Boeing                 177
BELLANCA               158
AERONCA                149
MAULE                  144
Mooney                 125
Air Tractor            117
AIRBUS                 110
EMBRAER                104
LUSCOMBE                95
CHAMPION                91
Name: count, dtype: int64

In [27]:
# Check the number of unique aircraft makes in the dataset
aviation_accidents_df['make'].nunique()

1309

### Claening Tasks

The make column represents the aircraft manufacturer. Upon inspection, several data quality issues were identified:

1. Inconsistent capitalization

Example: "Boeing", "BOEING", "boeing"

2. Leading/trailing whitespace

Some entries contain unnecessary spaces that affect grouping

3. Duplicate representations of the same manufacturer

Variations in spelling or formatting may split counts incorrectly

4. Rare manufacturers with very few observations

These can introduce noise and reduce statistical reliability

To address these issues, the column will be standardized and filtered.

In [28]:
# Standardize the 'make' column by converting all values to uppercase and stripping leading/trailing whitespace
aviation_accidents_df['make'] = (
    aviation_accidents_df['make']
    .astype(str)
    .str.replace(' ', '_')
    .str.strip()
    .str.upper()
)

In [29]:
# Standardize the 'make' column by converting all values to uppercase and stripping leading/trailing whitespace
aviation_accidents_df['make'] = aviation_accidents_df['make'].str.upper().str.strip()

In [30]:
# Inspect again the top 20 most common aircraft makes in the dataset after standardization
aviation_accidents_df['make'].value_counts().head(20)

make
CESSNA                7064
PIPER                 3960
BEECH                 1412
BOEING                 819
MOONEY                 360
CIRRUS_DESIGN_CORP     220
AIR_TRACTOR_INC        219
BELLANCA               219
MAULE                  215
AIR_TRACTOR            203
AERONCA                200
CHAMPION               158
GRUMMAN                147
LUSCOMBE               141
CIRRUS                 133
AIRBUS                 131
EMBRAER                130
STINSON                129
NORTH_AMERICAN         105
DEHAVILLAND             95
Name: count, dtype: int64

In [31]:
# Filter the dataset to include only rows with valid aircraft makes (appearing at least 50 times)
make_counts = aviation_accidents_df['make'].value_counts()

valid_makes = make_counts[make_counts >= 50].index

aviation_accidents_df = aviation_accidents_df[
    aviation_accidents_df['make'].isin(valid_makes)
]

In [32]:
# Verify the number of unique makes after filtering
aviation_accidents_df['make'].nunique()

35

In [33]:
# Display the top 5 most common aircraft makes in the dataset after filtering
aviation_accidents_df['make'].value_counts().head()

make
CESSNA    7064
PIPER     3960
BEECH     1412
BOEING     819
MOONEY     360
Name: count, dtype: int64

After cleaning, the make column was standardized by removing whitespace and converting all values to uppercase to ensure consistency across entries.

Additionally, manufacturers with fewer than 50 recorded incidents were excluded from the analysis. This threshold was chosen to ensure statistical reliability when comparing safety metrics across manufacturers.

By filtering out low-frequency manufacturers, the analysis focuses on makes with sufficient data to support meaningful and robust conclusions.

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [34]:
# Display the first few rows of the 'model' column to inspect its values and check for missing values
aviation_accidents_df['model'].head() 

4150          747
4171    PA-28-140
4285        DHC-6
6760      727-200
6806          C35
Name: model, dtype: object

In [35]:
# Check for missing values in the 'model' column to understand the extent of missing data in this column
aviation_accidents_df['model'].isna().sum() 

np.int64(10)

In [36]:
# Check the number of unique aircraft models in the dataset to understand the diversity of aircraft models involved in accidents
aviation_accidents_df['model'].nunique() 

1931

The model column represents the specific aircraft model. Initial inspection shows the presence of missing values, which must be addressed before analysis. Since the model is a key identifier for aircraft type, rows with missing model values cannot be reliably used and will be removed.

### Remove missing vallues

In [37]:
aviation_accidents_df = aviation_accidents_df[
    aviation_accidents_df['model'].notna()
]

In [38]:
# Standardize the 'model' column by converting all values to uppercase and stripping leading/trailing whitespace
aviation_accidents_df['model'] = (
    aviation_accidents_df['model']
    .astype(str)
    .str.replace(' ', '_')
    .str.strip()
    .str.lower()
)

In [39]:
# Display the top 20 most common aircraft models in the dataset after standardization
aviation_accidents_df['model'].value_counts().head(20)

model
172          757
152          312
182          301
172s         272
pa28         267
172n         246
sr22         238
180          213
737          199
a36          181
172m         179
150          176
pa-18-150    175
pa-28-140    169
172p         142
140          117
170b         107
172r         107
pa-28-180    105
pa-28-161    102
Name: count, dtype: int64

In [40]:
#Check if Models Are Unique Across Makes
aviation_accidents_df.groupby('make')['model'].nunique().sort_values(ascending=False).head(20)


make
CESSNA               394
PIPER                353
BEECH                222
BOEING               220
MAULE                 68
EMBRAER               63
NORTH_AMERICAN        48
AIRBUS                47
GRUMMAN               41
AIR_TRACTOR           39
MCDONNELL_DOUGLAS     38
DEHAVILLAND           37
AERO_COMMANDER        36
BELLANCA              36
AERONCA               34
DE_HAVILLAND          31
AIR_TRACTOR_INC       29
MOONEY                28
SOCATA                25
TAYLORCRAFT           25
Name: model, dtype: int64

Analysis shows that some model names are not unique to a single manufacturer. This means the same model name can appear under different makes, which creates ambiguity when identifying aircraft types.

Therefore, using the model column alone is insufficient to uniquely identify an aircraft.

In [41]:
# Create a new column 'make_model' by concatenating the 'make' and 'model' columns to create a unique identifier for each aircraft make and model combination
aviation_accidents_df['make_model'] = (
    aviation_accidents_df['make'] + "_" +
    aviation_accidents_df['model']
)

In [42]:
# Get the number of unique make_model combinations in the dataset to understand the diversity of aircraft make and model combinations involved in accidents
aviation_accidents_df['make_model'].nunique() 

2031

In [43]:
# Display the top 5 most common make_model combinations in the dataset to identify which specific aircraft make and model combinations are most frequently involved in accidents
aviation_accidents_df['make_model'].value_counts().head(5) 

make_model
CESSNA_172     757
CESSNA_152     312
CESSNA_182     301
CESSNA_172s    272
PIPER_pa28     267
Name: count, dtype: int64

To resolve ambiguity in aircraft identification, a new variable make_model was created by combining the manufacturer (make) and model name (model).

This ensures that each aircraft type is uniquely identified, even in cases where model names are shared across different manufacturers.

This derived feature is critical for accurate grouping and comparison of aircraft safety performance in later analysis.

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [44]:
# Inspect Columns of Interest for Analysis
cols = [
    'engine_type',
    'weather_condition',
    'number_of_engines',
    'purpose_of_flight',
    'broad_phase_of_flight'
]
aviation_accidents_df[cols].isna().sum()

engine_type               2614
weather_condition         1727
number_of_engines         1549
purpose_of_flight         2297
broad_phase_of_flight    14620
dtype: int64

In [45]:
aviation_accidents_df[cols].head(10) # Display the first few rows of the columns of interest to check for missing values and get an overview of the data in these columns

,engine_type,weather_condition,number_of_engines,purpose_of_flight,broad_phase_of_flight
4150,Turbo Fan,VMC,4.0,NaN,Taxi
4171,Reciprocating,IMC,1.0,Personal,Cruise
4285,Turbo Prop,VMC,2.0,Skydiving,Standing
6760,Turbo Fan,VMC,3.0,Unknown,Taxi
6806,Reciprocating,VMC,1.0,Personal,Climb
7084,Reciprocating,VMC,1.0,Personal,Takeoff
7708,Turbo Prop,VMC,2.0,Unknown,Landing
8585,Reciprocating,VMC,2.0,Personal,Approach
8865,Reciprocating,VMC,1.0,Aerial Application,Maneuvering
10247,Reciprocating,VMC,1.0,Personal,Approach


### 1. Engine.Type

In [46]:
# Display the distribution of values in the 'engine_type' column to understand the different types of engines involved in accidents and their frequency
aviation_accidents_df['engine_type'].value_counts(dropna=False)

engine_type
Reciprocating      12814
NaN                 2614
Turbo Prop           915
Turbo Fan            571
Unknown               79
Turbo Jet             64
Turbo Shaft            8
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64

In [47]:
# Standardize the 'engine_type' column by converting all values to lowercase and stripping leading/trailing whitespace
aviation_accidents_df['engine_type'] = (
    aviation_accidents_df['engine_type']
    .astype(str)
    .str.replace(' ', '_')
    .str.strip()
    .str.lower()
)

In [48]:
# Check after standardization the distribution of values in the 'engine_type' column to ensure that the standardization process has been effective in creating consistent categories for engine types
aviation_accidents_df['engine_type'].value_counts(dropna=False)

engine_type
reciprocating      12814
nan                 2614
turbo_prop           915
turbo_fan            571
unknown               79
turbo_jet             64
turbo_shaft            8
geared_turbofan        1
unk                    1
Name: count, dtype: int64

The engine_type column was standardized by converting all entries to uppercase and removing extra whitespace to ensure consistent grouping of engine categories.

### 2. Weather Condition

In [49]:
aviation_accidents_df['weather_condition'].value_counts(dropna=False) # Display the distribution of values in the 'weather_condition' column to understand the different weather conditions involved in accidents and their frequency

weather_condition
VMC    14226
NaN     1727
IMC      891
Unk      167
UNK       56
Name: count, dtype: int64

In [50]:
# Handle missing values in the 'weather_condition' column by replacing 'UNK' and 'UNKNOWN' with NaN to indicate that the weather condition is unknown or not specified
aviation_accidents_df['weather_condition'] = (
    aviation_accidents_df['weather_condition']
    .replace(['UNK', 'UNKNOWN'], np.nan)
)

Placeholder values such as "UNK" and "UNKNOWN" were replaced with missing values (NaN) to better reflect the absence of reliable weather data.

### 3. Number of Engines

In [51]:
# Inspect
aviation_accidents_df['number_of_engines'].value_counts(dropna=False) # Display the distribution of values in the 'number_of_engines' column to understand the different engine configurations involved in accidents and their frequency

number_of_engines
1.0    13184
2.0     2257
NaN     1549
4.0       50
3.0       23
0.0        4
Name: count, dtype: int64

In [52]:
# Cleaning the 'number_of_engines' column by replacing non-numeric values with NaN and converting the column to numeric type for analysis
aviation_accidents_df['number_of_engines'] = pd.to_numeric(
    aviation_accidents_df['number_of_engines'], errors='coerce'
)

The number_of_engines column was converted to a numeric format. Invalid or non-numeric values were coerced to NaN, ensuring the column can be used in quantitative analysis.

### 4. Purpose of Flight

In [53]:
# Inspect the 'purpose_of_flight' column to understand the different purposes of flight involved in accidents and their frequency
aviation_accidents_df['purpose_of_flight'].value_counts(dropna=False)

purpose_of_flight
Personal                     9830
Instructional                2405
NaN                          2297
Aerial Application            723
Business                      406
Unknown                       270
Positioning                   266
Skydiving                     157
Aerial Observation            146
Other Work Use                120
Banner Tow                     86
Flight Test                    72
Ferry                          72
Executive/corporate            65
Glider Tow                     29
Public Aircraft - Federal      28
Public Aircraft                27
Public Aircraft - State        21
Air Race show                  15
Firefighting                   12
Public Aircraft - Local        11
PUBS                            3
Air Race/show                   2
Air Drop                        2
ASHO                            2
Name: count, dtype: int64

In [54]:
# Standardize purpose of flight by converting all values to lowercase and stripping leading/trailing whitespace
aviation_accidents_df['purpose_of_flight'] = (
    aviation_accidents_df['purpose_of_flight']
    .astype(str)
    .str.replace(' ', '_')
    .str.strip()
    .str.lower()
)

In [55]:
# Check after standardization
aviation_accidents_df['purpose_of_flight'].value_counts(dropna=False) # Display the distribution of values in the 'purpose_of_flight' column after standardization to ensure that the standardization process has been effective in creating consistent categories for purposes of flight

purpose_of_flight
personal                     9830
instructional                2405
nan                          2297
aerial_application            723
business                      406
unknown                       270
positioning                   266
skydiving                     157
aerial_observation            146
other_work_use                120
banner_tow                     86
flight_test                    72
ferry                          72
executive/corporate            65
glider_tow                     29
public_aircraft_-_federal      28
public_aircraft                27
public_aircraft_-_state        21
air_race_show                  15
firefighting                   12
public_aircraft_-_local        11
pubs                            3
air_race/show                   2
air_drop                        2
asho                            2
Name: count, dtype: int64

The purpose_of_flight column was standardized by converting text to uppercase and removing whitespace to ensure consistency across categories.

### 5. Phase of Flight

In [56]:
aviation_accidents_df['broad_phase_of_flight'].value_counts(dropna=False)

broad_phase_of_flight
NaN            14620
Landing         1108
Takeoff          424
Cruise           238
Approach         210
Maneuvering      127
Taxi              99
Go-around         81
Descent           62
Climb             52
Standing          34
Unknown           10
Other              2
Name: count, dtype: int64

In [57]:
# Clean and standardize the 'broad_phase_of_flight' column by converting all values to lowercase, replacing spaces with underscores, and stripping leading/trailing whitespace to create consistent categories for phases of flight
aviation_accidents_df['broad_phase_of_flight'] = (
    aviation_accidents_df['broad_phase_of_flight']
    .astype(str)
    .str.replace(' ', '_')
    .str.strip()
    .str.lower()
)

In [58]:
aviation_accidents_df['broad_phase_of_flight'].value_counts(dropna=False)

broad_phase_of_flight
nan            14620
landing         1108
takeoff          424
cruise           238
approach         210
maneuvering      127
taxi              99
go-around         81
descent           62
climb             52
standing          34
unknown           10
other              2
Name: count, dtype: int64

In [59]:
# Replace unkown values
aviation_accidents_df['broad_phase_of_flight'] = aviation_accidents_df[
    'broad_phase_of_flight'
].replace(['UNKNOWN'], np.nan)

The broad_phase_of_flight column was cleaned by standardizing text formatting and handling placeholder values. This ensures accurate grouping when analyzing accident severity across different flight phases

In [60]:
aviation_accidents_df[cols].describe(include='all')

,engine_type,weather_condition,number_of_engines,purpose_of_flight,broad_phase_of_flight
count,17067,15284,15518.000000,17067,17067
unique,9,3,NaN,25,13
top,reciprocating,VMC,NaN,personal,nan
freq,12814,14226,NaN,9830,14620
mean,NaN,NaN,1.157817,NaN,NaN
std,NaN,NaN,0.394630,NaN,NaN
min,NaN,NaN,0.000000,NaN,NaN
25%,NaN,NaN,1.000000,NaN,NaN
50%,NaN,NaN,1.000000,NaN,NaN
75%,NaN,NaN,1.000000,NaN,NaN


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [61]:
# Inspect missing values
aviation_accidents_df.isna().sum().sort_values(ascending=False)

schedule                  15318
air_carrier                9055
airport_code               5635
airport_name               5536
report_status              3115
weather_condition          1783
number_of_engines          1549
longitude                  1288
latitude                   1285
aircraft_damage             664
publication_date            581
far_description             221
registration_number         130
location                      4
country                       1
event_id                      0
event_date                    0
accident_number               0
investigation_type            0
injury_severity               0
engine_type                   0
amateur_built                 0
make                          0
model                         0
aircraft_category             0
total_serious_injuries        0
total_fatal_injuries          0
purpose_of_flight             0
total_uninjured               0
total_minor_injuries          0
broad_phase_of_flight         0
total_pa

In [62]:
# Step 2: Check non-null counts
missing_percent = aviation_accidents_df.isna().mean() * 100
missing_percent.sort_values(ascending=False).round(2)

schedule                  89.75
air_carrier               53.06
airport_code              33.02
airport_name              32.44
report_status             18.25
weather_condition         10.45
number_of_engines          9.08
longitude                  7.55
latitude                   7.53
aircraft_damage            3.89
publication_date           3.40
far_description            1.29
registration_number        0.76
location                   0.02
country                    0.01
event_id                   0.00
event_date                 0.00
accident_number            0.00
investigation_type         0.00
injury_severity            0.00
engine_type                0.00
amateur_built              0.00
make                       0.00
model                      0.00
aircraft_category          0.00
total_serious_injuries     0.00
total_fatal_injuries       0.00
purpose_of_flight          0.00
total_uninjured            0.00
total_minor_injuries       0.00
broad_phase_of_flight      0.00
total_pa

Instead of using absolute counts, missing values were evaluated as a percentage of total rows. This provides a normalized measure of data completeness and allows fair comparison across columns.

In [63]:
threshold = 50  # percent missing allowed

In [64]:
# Identify columns to drop based on the threshold
cols_to_drop = missing_percent[missing_percent > threshold].index.tolist()
cols_to_drop

['schedule', 'air_carrier']

In [65]:
#Drop columns with more than 50% missing values
aviation_accidents_df = aviation_accidents_df.drop(columns=cols_to_drop)

In [66]:
# Verify after dropping columns with more than 50% missing values
aviation_accidents_df.shape

aviation_accidents_df.isna().mean().sort_values(ascending=False).head(10)

airport_code         0.330169
airport_name         0.324369
report_status        0.182516
weather_condition    0.104471
number_of_engines    0.090760
longitude            0.075467
latitude             0.075291
aircraft_damage      0.038905
publication_date     0.034042
far_description      0.012949
dtype: float64

Columns with more than 50% missing values were removed from the dataset. This percentage-based approach ensures that only features with sufficient data coverage are retained, improving the reliability and robustness of the analysis.

Using a relative threshold rather than absolute counts allows the method to scale appropriately with dataset size and ensures consistency in feature selection.

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [67]:
# Save the cleaned dataset to a new CSV file for future analysis
aviation_accidents_df.to_csv("data/aviation_accidents_cleaned.csv", index=False)